# Stage 1 — Linear Probe Baseline

This Colab notebook evaluates frozen **ImageNet-1K ResNet-18** and **DINOv2 ViT-S/14** features with a learned multiclass linear head on **DTD partition 1**, **FGVC-Aircraft (variant)**, and **Oxford Flowers-102**. It preserves official train/validation/test splits, caches features once, evaluates 5-shot, 10-shot, and full-data settings, and reports the Stage 1 metrics and visualizations. No Flow Matching component is used.

Run all cells from top to bottom. The full grid is compute-intensive; cached features make repeat runs much faster.

## 1. Install dependencies

In [1]:
%pip -q install open_clip_torch scikit-learn pandas seaborn tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00


## 2. Mount Drive and configure paths
Set `USE_DRIVE=False` to use temporary Colab storage. All artifacts are grouped below `OUTPUT_ROOT`.

In [2]:
from pathlib import Path
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Flow-Matching')
else:
    PROJECT_ROOT = Path('/content/Flow-Matching')
DATA_ROOT = PROJECT_ROOT / 'data'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'linear_probe'
CACHE_ROOT = PROJECT_ROOT / 'feature_cache'
for p in [DATA_ROOT, OUTPUT_ROOT, CACHE_ROOT]: p.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


Mounted at /content/drive
Project root: /content/drive/MyDrive/Flow-Matching


## 3. Imports, hardware, and reproducibility
Subset seeds select few-shot examples. Full-data repetitions use classifier-initialization seeds. The test split is never used for checkpoint selection.

In [3]:
import copy, json, math, os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import open_clip

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if PIN_MEMORY else 'none')

DATASETS = ['dtd', 'aircraft', 'flowers102']
ENCODERS = ['resnet18', 'dinov2_vits14', 'clip_rn50']
SHOT_SETTINGS = [5, 10, 'full']
SUBSET_SEEDS = [0, 1, 2]
INIT_SEEDS = [0, 1, 2]
# Conservative image batch size: DINOv2/CLIP can exceed Colab GPU memory at larger values.
BATCH_SIZE_IMAGES = 32
BATCH_SIZE_HEAD = 64
MAX_EPOCHS = 200
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
# Zero workers is more reliable in Colab notebooks and avoids worker-process crashes.
NUM_WORKERS = 0
CHECKPOINT_INTERVAL = 25
RESUME_TRAINING = True

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(0)
CONFIG = {k: v for k, v in dict(datasets=DATASETS, encoders=ENCODERS, shots=SHOT_SETTINGS, subset_seeds=SUBSET_SEEDS, init_seeds=INIT_SEEDS, max_epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE_HEAD, optimizer='AdamW', checkpoint_interval=CHECKPOINT_INTERVAL, resume_training=RESUME_TRAINING).items()}
(OUTPUT_ROOT / 'config.json').write_text(json.dumps(CONFIG, indent=2))


Device: cuda | GPU: Tesla T4


446

## 4. Official datasets and frozen encoders
Torchvision supplies the official splits. DTD explicitly uses partition 1 and Aircraft explicitly uses the `variant` label level. Each encoder uses its checkpoint-associated preprocessing.

In [4]:
def build_dataset(name, split, transform):
    if name == 'dtd':
        return datasets.DTD(DATA_ROOT, split=split, partition=1, transform=transform, download=True)
    if name == 'aircraft':
        return datasets.FGVCAircraft(DATA_ROOT, split=split, annotation_level='variant', transform=transform, download=True)
    if name == 'flowers102':
        return datasets.Flowers102(DATA_ROOT, split=split, transform=transform, download=True)
    raise ValueError(name)

def targets_of(ds):
    for attr in ('_labels', 'targets', 'labels'):
        if hasattr(ds, attr): return np.asarray(getattr(ds, attr), dtype=int)
    return np.asarray([ds[i][1] for i in range(len(ds))], dtype=int)

def class_names_of(ds):
    if hasattr(ds, 'classes'): return [str(x).replace('_', ' ') for x in ds.classes]
    n = int(targets_of(ds).max()) + 1
    return [f'class {i + 1}' for i in range(n)]

def load_encoder(name):
    if name == 'resnet18':
        weights = ResNet18_Weights.IMAGENET1K_V1
        net = models.resnet18(weights=weights)
        net.fc = nn.Identity()
        preprocess = weights.transforms()
        dim = 512
    elif name == 'dinov2_vits14':
        try:
            net = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True)
        except TypeError:
            # Older torch versions do not accept trust_repo.
            net = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        preprocess = transforms.Compose([transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
        dim = 384
    elif name == 'clip_rn50':
        net, _, preprocess = open_clip.create_model_and_transforms('RN50', pretrained='openai')
        class CLIPImageEncoder(nn.Module):
            def __init__(self, clip_model): super().__init__(); self.clip_model = clip_model
            def forward(self, x): return self.clip_model.encode_image(x)
        net = CLIPImageEncoder(net)
        dim = 1024
    else: raise ValueError(name)
    net.eval().to(DEVICE)
    for p in net.parameters(): p.requires_grad_(False)
    assert not any(p.requires_grad for p in net.parameters())
    return net, preprocess, dim

def load_cached(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        # Compatibility with older PyTorch releases.
        return torch.load(path, map_location='cpu')

@torch.inference_mode()
def extract_split(dataset_name, split, encoder_name, encoder, preprocess):
    cache = CACHE_ROOT / f'{dataset_name}__{encoder_name}__{split}.pt'
    if cache.exists(): return load_cached(cache)
    ds = build_dataset(dataset_name, split, preprocess)
    loader = DataLoader(ds, batch_size=BATCH_SIZE_IMAGES, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    features, labels = [], []
    for images, y in tqdm(loader, desc=f'{dataset_name}/{encoder_name}/{split}'):
        z = encoder(images.to(DEVICE, non_blocking=True)).float().cpu()
        features.append(z); labels.append(y.long())
    item = dict(features=torch.cat(features), labels=torch.cat(labels), classes=class_names_of(ds), dataset=dataset_name, split=split, encoder=encoder_name)
    assert len(item['features']) == len(ds) == len(item['labels'])
    assert torch.isfinite(item['features']).all()
    torch.save(item, cache)
    return item


## 5. Extract and cache train/validation/test features
This is the expensive encoder pass. Re-running the notebook loads the cache instead.

In [5]:
feature_bank = {}
for encoder_name in ENCODERS:
    print(f'\nLoading frozen encoder: {encoder_name}')
    encoder, preprocess, feature_dim = load_encoder(encoder_name)
    try:
        for dataset_name in DATASETS:
            print(f'  Extracting {dataset_name} ({feature_dim} dimensions)')
            feature_bank[(dataset_name, encoder_name)] = {split: extract_split(dataset_name, split, encoder_name, encoder, preprocess) for split in ('train', 'val', 'test')}
    except RuntimeError as error:
        if 'out of memory' in str(error).lower() and torch.cuda.is_available():
            torch.cuda.empty_cache()
            raise RuntimeError(f'CUDA out of memory while extracting {encoder_name}. Reduce BATCH_SIZE_IMAGES below {BATCH_SIZE_IMAGES} and rerun this cell.') from error
        raise RuntimeError(f'Feature extraction failed for encoder={encoder_name}. The first error above identifies the dataset/split.') from error
    finally:
        del encoder
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Cached combinations:', list(feature_bank))


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 126MB/s] 
100%|██████████| 625M/625M [00:26<00:00, 23.8MB/s] 


dtd/resnet18/train:   0%|          | 0/15 [00:00<?, ?it/s]

dtd/resnet18/val:   0%|          | 0/15 [00:00<?, ?it/s]

dtd/resnet18/test:   0%|          | 0/15 [00:00<?, ?it/s]

...

flowers102/clip_rn50/train:   0%|          | 0/8 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x791b4258b1a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
AssertionError<function _MultiProcessingDataLoaderIter.__del__ at 0x791b4258b1a0>: can only test a child process
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x791b4258b1a0>self._shutdown_workers()
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-

flowers102/clip_rn50/val:   0%|          | 0/8 [00:00<?, ?it/s]

flowers102/clip_rn50/test:   0%|          | 0/49 [00:00<?, ?it/s]

Cached combinations: [('dtd', 'resnet18'), ('aircraft', 'resnet18'), ('flowers102', 'resnet18'), ('dtd', 'dinov2_vits14'), ('aircraft', 'dinov2_vits14'), ('flowers102', 'dinov2_vits14'), ('dtd', 'clip_rn50'), ('aircraft', 'clip_rn50'), ('flowers102', 'clip_rn50')]


## 6. Few-shot manifests and linear-head training
The best checkpoint is selected by full validation accuracy. For 5/10-shot runs, the run seed controls the balanced subset; for full-data runs it controls only classifier initialization.

In [ ]:
def balanced_indices(labels, k, seed):
    labels = np.asarray(labels); rng = np.random.default_rng(seed); chosen = []
    for c in np.unique(labels):
        idx = np.flatnonzero(labels == c)
        if len(idx) < k: raise ValueError(f'class {c} has {len(idx)} samples, fewer than K={k}')
        chosen.extend(rng.choice(idx, size=k, replace=False).tolist())
    return np.asarray(sorted(chosen))

def eval_head(head, x, y, criterion):
    head.eval(); total_loss = 0.0; predictions = []
    loader = DataLoader(TensorDataset(x, y), batch_size=512, shuffle=False)
    with torch.inference_mode():
        for xb, yb in loader:
            logits = head(xb.to(DEVICE)); total_loss += criterion(logits, yb.to(DEVICE)).item() * len(yb)
            predictions.append(logits.argmax(1).cpu())
    pred = torch.cat(predictions)
    return total_loss / len(y), (pred == y).float().mean().item(), pred

def train_linear_run(bank, dataset_name, encoder_name, shot, run_seed):
    seed_everything(run_seed)
    x_train, y_train = bank['train']['features'].float(), bank['train']['labels'].long()
    if shot != 'full':
        idx = torch.as_tensor(balanced_indices(y_train.numpy(), int(shot), run_seed))
        x_train, y_train = x_train[idx], y_train[idx]
    x_val, y_val = bank['val']['features'].float(), bank['val']['labels'].long()
    x_test, y_test = bank['test']['features'].float(), bank['test']['labels'].long()
    n_classes = len(bank['train']['classes']); head = nn.Linear(x_train.shape[1], n_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(); best_acc = -1.0; best_state = None; history = []; start_epoch = 1
    run_dir = OUTPUT_ROOT / 'runs' / dataset_name / encoder_name / str(shot) / f'seed_{run_seed}'
    checkpoint_dir = run_dir / 'checkpoints'; checkpoint_dir.mkdir(parents=True, exist_ok=True)
    latest_path, best_path = checkpoint_dir / 'latest.pt', checkpoint_dir / 'best.pt'
    run_config = dict(CONFIG, dataset=dataset_name, encoder=encoder_name, shot=str(shot), run_seed=run_seed, input_dim=x_train.shape[1], num_classes=n_classes)
    (run_dir / 'config.json').write_text(json.dumps(run_config, indent=2))
    if RESUME_TRAINING and latest_path.exists():
        checkpoint = torch.load(latest_path, map_location=DEVICE, weights_only=False)
        head.load_state_dict(checkpoint['model_state_dict']); optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1; best_acc = checkpoint['best_val_accuracy']; best_state = checkpoint['best_model_state_dict']; history = checkpoint['history']
        print(f'Resuming {run_dir} from epoch {start_epoch}')
    if DEVICE.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    def save_checkpoint(path, epoch):
        torch.save(dict(epoch=epoch, model_state_dict=head.state_dict(), optimizer_state_dict=optimizer.state_dict(), best_model_state_dict=best_state, best_val_accuracy=best_acc, history=history, config=run_config, torch_rng_state=torch.get_rng_state(), cuda_rng_state=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None), path)
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        generator = torch.Generator().manual_seed(run_seed * 100000 + epoch)
        loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE_HEAD, shuffle=True, generator=generator)
        head.train(); train_total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); optimizer.zero_grad(set_to_none=True)
            loss = criterion(head(xb), yb); loss.backward(); optimizer.step(); train_total += loss.item() * len(yb)
        val_loss, val_acc, _ = eval_head(head, x_val, y_val, criterion)
        history.append(dict(epoch=epoch, train_loss=train_total/len(y_train), val_loss=val_loss, val_accuracy=val_acc))
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(head.state_dict()); save_checkpoint(best_path, epoch)
        save_checkpoint(latest_path, epoch)
        if epoch % CHECKPOINT_INTERVAL == 0: save_checkpoint(checkpoint_dir / f'epoch_{epoch:04d}.pt', epoch)
    if best_state is None: best_state = copy.deepcopy(head.state_dict())
    runtime_seconds = time.perf_counter() - started
    head.load_state_dict(best_state)
    test_loss, test_acc, test_pred = eval_head(head, x_test, y_test, criterion)
    torch.save(dict(epoch=MAX_EPOCHS, model_state_dict=head.state_dict(), optimizer_state_dict=optimizer.state_dict(), best_val_accuracy=best_acc, history=history, config=run_config), checkpoint_dir / 'final.pt')
    torch.save(dict(state_dict=head.state_dict(), input_dim=x_train.shape[1], num_classes=n_classes, config=run_config), run_dir / 'best_linear_head.pt')
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    np.save(run_dir / 'test_predictions.npy', test_pred.numpy())
    peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == 'cuda' else 0.0
    metrics = dict(dataset=dataset_name, encoder=encoder_name, shot=str(shot), seed=run_seed, best_val_accuracy=best_acc, test_accuracy=test_acc, test_loss=test_loss, runtime_seconds=runtime_seconds, peak_gpu_memory_mb=peak_gpu_memory_mb, resumed_from_epoch=start_epoch)
    (run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    return metrics, history, test_pred


## 7. Run the complete experiment grid
This performs 54 trainings: 3 datasets × 2 encoders × 3 training sizes × 3 repetitions.

In [ ]:
all_results, histories, predictions = [], {}, {}
for dataset_name in DATASETS:
    for encoder_name in ENCODERS:
        bank = feature_bank[(dataset_name, encoder_name)]
        for shot in SHOT_SETTINGS:
            seeds = SUBSET_SEEDS if shot != 'full' else INIT_SEEDS
            for run_seed in seeds:
                print(f'\n{dataset_name} | {encoder_name} | {shot} | seed {run_seed}')
                metrics, history, pred = train_linear_run(bank, dataset_name, encoder_name, shot, run_seed)
                all_results.append(metrics); histories[(dataset_name, encoder_name, str(shot), run_seed)] = history; predictions[(dataset_name, encoder_name, str(shot), run_seed)] = pred
results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_ROOT / 'run_metrics.csv', index=False)
display(results_df)


## 8. Aggregate accuracy and plot training-size curves

In [ ]:
summary = results_df.groupby(['dataset','encoder','shot'], as_index=False).agg(mean_accuracy=('test_accuracy','mean'), std_accuracy=('test_accuracy','std'), runs=('test_accuracy','size'))
summary.to_csv(OUTPUT_ROOT / 'accuracy_summary.csv', index=False)
display(summary.style.format({'mean_accuracy':'{:.4f}', 'std_accuracy':'{:.4f}'}))
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, dataset_name in zip(axes, DATASETS):
    part = summary[summary.dataset == dataset_name].copy(); part['x'] = part.shot.map({'5':0,'10':1,'full':2})
    for encoder_name in ENCODERS:
        row = part[part.encoder == encoder_name].sort_values('x')
        ax.errorbar(row.x, row.mean_accuracy, yerr=row.std_accuracy.fillna(0), marker='o', capsize=4, label=encoder_name)
    ax.set(title=dataset_name, xticks=[0,1,2], xticklabels=['5','10','full'], xlabel='training images per class', ylabel='top-1 test accuracy'); ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'accuracy_vs_training_size.png', dpi=180, bbox_inches='tight'); plt.show()


## 9. Representative 10-shot training/validation loss curves
Seed 0 is used consistently as the representative run for each dataset-encoder pair.

In [ ]:
fig, axes = plt.subplots(len(DATASETS), len(ENCODERS), figsize=(13, 11), squeeze=False)
for i, dataset_name in enumerate(DATASETS):
    for j, encoder_name in enumerate(ENCODERS):
        h = pd.DataFrame(histories[(dataset_name, encoder_name, '10', 0)])
        axes[i,j].plot(h.epoch, h.train_loss, label='train'); axes[i,j].plot(h.epoch, h.val_loss, label='validation')
        axes[i,j].set(title=f'{dataset_name} / {encoder_name}', xlabel='epoch', ylabel='cross-entropy'); axes[i,j].legend(); axes[i,j].grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'representative_loss_curves.png', dpi=180, bbox_inches='tight'); plt.show()


## 10. Row-normalized confusion matrices
For each dataset, the representative setting is DINOv2, 10-shot, seed 0.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, dataset_name in zip(axes, DATASETS):
    bank = feature_bank[(dataset_name, 'dinov2_vits14')]; y = bank['test']['labels'].numpy(); pred = predictions[(dataset_name, 'dinov2_vits14', '10', 0)].numpy()
    cm = confusion_matrix(y, pred, normalize='true'); sns.heatmap(cm, cmap='mako', vmin=0, vmax=1, ax=ax, cbar=False)
    ax.set(title=f'{dataset_name}: DINOv2 10-shot', xlabel='predicted class', ylabel='true class')
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'confusion_matrices.png', dpi=180, bbox_inches='tight'); plt.show()


## 11. Comparable feature visualizations
PCA is fitted jointly to the same deterministic subset of test examples for both encoders. The first 10 classes and up to 30 test examples per class are used; class colors are fixed within each dataset. These plots are qualitative.

In [ ]:
fig, axes = plt.subplots(len(DATASETS), len(ENCODERS), figsize=(13, 12), squeeze=False)
for i, dataset_name in enumerate(DATASETS):
    selected_classes = np.arange(10); base_labels = feature_bank[(dataset_name, 'resnet18')]['test']['labels'].numpy(); selected_idx = np.concatenate([np.flatnonzero(base_labels == c)[:30] for c in selected_classes])
    for j, encoder_name in enumerate(ENCODERS):
        bank = feature_bank[(dataset_name, encoder_name)]['test']; z = bank['features'][selected_idx].numpy(); y = bank['labels'][selected_idx].numpy(); xy = PCA(n_components=2).fit_transform(z)
        sns.scatterplot(x=xy[:,0], y=xy[:,1], hue=y, palette='tab10', s=22, alpha=.75, ax=axes[i,j], legend=(j == 1))
        axes[i,j].set(title=f'{dataset_name} / {encoder_name}', xlabel='PC1', ylabel='PC2')
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'feature_pca.png', dpi=180, bbox_inches='tight'); plt.show()
print('Artifacts saved to:', OUTPUT_ROOT)
